# Customer Churn Prediction & Customer Segmentation

## 03 — Data Preparation

### Data Cleaning + Feature Engineering + Model-Ready Dataset

The previous notebooks helped us understand:

- whether the raw data is structurally usable
- where data-quality issues exist
- which customer characteristics appear to be associated with churn

Now we prepare the dataset for machine learning.

### What this notebook will do

1. Load the raw dataset again.
2. Clean `TotalCharges`.
3. Validate the cleaning result.
4. Remove the customer identifier from modelling features.
5. Convert the target into a machine-learning friendly format.
6. Create a small number of business-meaningful features.
7. Separate features and target.
8. Identify numerical and categorical features.
9. Create a train/test split.
10. Build a preprocessing pipeline.
11. Fit the preprocessing only on training data.
12. Transform validation/test data without leaking information.
13. Save the prepared datasets and preprocessing object for later modelling.

> **Principle:** Feature engineering should add useful information, not simply increase the number of columns.

## 1. Import libraries

We use Pandas for data preparation and Scikit-learn for reproducible preprocessing.

The preprocessing pipeline is important because the exact same transformations must later be applied to new customers in production.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Define project paths

We keep the raw data separate from processed data.

The raw dataset should never be overwritten during cleaning.

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Customer-Churn.csv"
)

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Raw dataset:", RAW_DATA_PATH)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Model directory:", MODEL_DIR)

Raw dataset: d:\Projects\Customer Churn Segmentation\data\raw\Customer-Churn.csv
Processed data directory: d:\Projects\Customer Churn Segmentation\data\processed
Model directory: d:\Projects\Customer Churn Segmentation\models


## 3. Load the raw dataset

We start from the original raw file rather than using an already modified DataFrame.

This makes the preparation process reproducible from scratch.

In [3]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset was not found at: {RAW_DATA_PATH}"
    )

df = pd.read_csv(RAW_DATA_PATH)

print(
    f"Raw dataset loaded successfully: "
    f"{df.shape[0]:,} rows and {df.shape[1]} columns."
)

Raw dataset loaded successfully: 7,043 rows and 21 columns.


## 4. Make a working copy

We never modify the raw object directly.

The `working_df` DataFrame is the version we are allowed to transform during this notebook.

In [4]:
working_df = df.copy()

print("Working copy created.")

Working copy created.


# Part A — Data Cleaning

## 5. Check the raw `TotalCharges` column

During validation we discovered that `TotalCharges` is stored as text even though it represents a monetary value.

Before converting it, we want to understand how many values are affected.

In [5]:
print("Original TotalCharges data type:")
print(working_df["TotalCharges"].dtype)

print()

print("Example values:")
display(working_df["TotalCharges"].head(10))

Original TotalCharges data type:
str

Example values:


0      29.85
1     1889.5
2     108.15
3    1840.75
4     151.65
5      820.5
6     1949.4
7      301.9
8    3046.05
9    3487.95
Name: TotalCharges, dtype: str

In [6]:
total_charges_numeric = pd.to_numeric(
    working_df["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

conversion_failures = total_charges_numeric.isnull().sum()

print(
    f"TotalCharges values that cannot be converted to numeric: "
    f"{conversion_failures:,}"
)

TotalCharges values that cannot be converted to numeric: 11


## 6. Inspect the records affected by `TotalCharges`

We should never fill or remove problematic records without first understanding them.

Let's inspect the customers whose `TotalCharges` could not be converted.

In [7]:
problematic_total_charges = working_df.loc[
    total_charges_numeric.isnull(),
    [
        "customerID",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "Churn"
    ]
]

display(problematic_total_charges)

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


### Cleaning decision

The problematic values in this dataset occur where the customer has a very short tenure and the raw `TotalCharges` field contains a blank-like value.

For a new customer, a missing total charge can reasonably be represented by the amount accumulated so far.

We therefore use:

**TotalCharges = tenure × MonthlyCharges**

for records where the original value cannot be converted.

This is a business-informed reconstruction rather than a generic mean/median imputation.

In [8]:
working_df["TotalCharges"] = total_charges_numeric

missing_total_charges_before_fix = (
    working_df["TotalCharges"].isnull().sum()
)

print(
    "Missing TotalCharges after numeric conversion:",
    missing_total_charges_before_fix
)

Missing TotalCharges after numeric conversion: 11


In [9]:
missing_total_charges_mask = working_df["TotalCharges"].isnull()

working_df.loc[
    missing_total_charges_mask,
    "TotalCharges"
] = (
    working_df.loc[
        missing_total_charges_mask,
        "tenure"
    ]
    * working_df.loc[
        missing_total_charges_mask,
        "MonthlyCharges"
    ]
)

print(
    "TotalCharges values reconstructed:",
    missing_total_charges_mask.sum()
)

TotalCharges values reconstructed: 11


## 7. Validate the cleaned `TotalCharges`

After cleaning, we should verify the result rather than assuming it worked.

In [10]:
remaining_missing_total_charges = (
    working_df["TotalCharges"].isnull().sum()
)

print(
    "Remaining missing TotalCharges values:",
    remaining_missing_total_charges
)

print()
print("TotalCharges data type:")
print(working_df["TotalCharges"].dtype)

Remaining missing TotalCharges values: 0

TotalCharges data type:
float64


In [11]:
if remaining_missing_total_charges == 0:
    print("TotalCharges cleaning completed successfully.")
else:
    print("TotalCharges still contains missing values and requires investigation.")

TotalCharges cleaning completed successfully.


## 8. Check for negative or impossible monetary values

Cleaning should not introduce invalid values.

We verify that both monthly and total charges are non-negative.

In [12]:
negative_monthly_charges = (
    working_df["MonthlyCharges"] < 0
).sum()

negative_total_charges = (
    working_df["TotalCharges"] < 0
).sum()

print(
    f"Negative MonthlyCharges: {negative_monthly_charges:,}"
)

print(
    f"Negative TotalCharges: {negative_total_charges:,}"
)

Negative MonthlyCharges: 0
Negative TotalCharges: 0


# Part B — Remove information that should not be a model feature

## 9. Separate the customer identifier

`customerID` is useful for:

- tracing predictions back to customers
- reporting
- debugging
- business actions

But the ID itself does not represent customer behaviour.

Therefore, we keep it available separately and exclude it from the model features.

In [13]:
customer_ids = working_df["customerID"].copy()

working_df = working_df.drop(
    columns=["customerID"]
)

print(
    "customerID removed from the modelling DataFrame."
)

customerID removed from the modelling DataFrame.


## 10. Validate the target column

The target is `Churn`.

For binary classification, we will convert:

- `No` → `0`
- `Yes` → `1`

The original meaning is preserved, but the model receives a numerical target.

In [14]:
target_column = "Churn"

print("Original target values:")
print(working_df[target_column].value_counts())

Original target values:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [15]:
target_mapping = {
    "No": 0,
    "Yes": 1
}

working_df[target_column] = (
    working_df[target_column]
    .map(target_mapping)
)

print("Target values after encoding:")
print(working_df[target_column].value_counts().sort_index())

Target values after encoding:
Churn
0    5174
1    1869
Name: count, dtype: int64


### Target interpretation

After encoding:

```text
0 = Customer did not churn
1 = Customer churned
```

This convention will be used consistently throughout the modelling pipeline.

# Part C — Feature Engineering

Feature engineering should reflect the business problem.

We will create only a few features where the underlying meaning is clear.

We are not going to generate dozens of arbitrary ratios or combinations just to increase the feature count.

## 11. Create `AverageMonthlySpend`

`TotalCharges` represents accumulated customer spending.

Dividing it by tenure can provide an approximate view of the customer's historical monthly spending.

However, new customers with zero tenure would create a division-by-zero problem.

We therefore use a safe calculation.

In [16]:
working_df["AverageMonthlySpend"] = (
    working_df["TotalCharges"]
    / working_df["tenure"].clip(lower=1)
)

print("AverageMonthlySpend feature created.")

AverageMonthlySpend feature created.


## 12. Create an early-lifecycle indicator

Our EDA suggested that customer lifecycle stage may be relevant to churn.

Instead of forcing the model to infer a simple business distinction from scratch, we create a readable indicator:

- `1` → customer has been with the company for 12 months or less
- `0` → customer has been with the company for more than 12 months

This is a business-oriented feature, not a random transformation.

In [17]:
working_df["IsNewCustomer"] = (
    working_df["tenure"] <= 12
).astype(int)

print("IsNewCustomer distribution:")
print(
    working_df["IsNewCustomer"]
    .value_counts()
    .sort_index()
)

IsNewCustomer distribution:
IsNewCustomer
0    4857
1    2186
Name: count, dtype: int64


## 13. Create a service count feature

Customers can have several optional services.

Instead of treating every service independently for one business view, we can create a simple count of optional services subscribed to.

This gives us a rough measure of **service adoption**.

We will keep the original service columns as well because the model may find individual services useful.

In [18]:
service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

service_count = pd.Series(
    0,
    index=working_df.index
)

for column in service_columns:
    service_count = service_count + (
        working_df[column] == "Yes"
    ).astype(int)

working_df["ServiceCount"] = service_count

print("ServiceCount created.")
display(
    working_df["ServiceCount"]
    .value_counts()
    .sort_index()
    .to_frame("Customers")
)

ServiceCount created.


,Customers
ServiceCount,
0,2219
1,966
2,1033
3,1118
4,852
5,571
6,284


### Important feature-engineering decision

We are keeping both:

- the individual service columns
- `ServiceCount`

Why?

Because a total count tells us how many services a customer has, while the individual columns tell us **which services** they have.

We will later let model evaluation determine whether all of these representations are useful.

# Part D — Final feature review

## 14. Inspect the prepared dataset

Before splitting the data, we should review the resulting columns and data types.

In [19]:
print(
    f"Prepared dataset shape: "
    f"{working_df.shape[0]:,} rows × {working_df.shape[1]} columns"
)

print()
print("Columns:")
for column_number, column_name in enumerate(
    working_df.columns,
    start=1
):
    print(f"{column_number:02d}. {column_name}")

Prepared dataset shape: 7,043 rows × 23 columns

Columns:
01. gender
02. SeniorCitizen
03. Partner
04. Dependents
05. tenure
06. PhoneService
07. MultipleLines
08. InternetService
09. OnlineSecurity
10. OnlineBackup
11. DeviceProtection
12. TechSupport
13. StreamingTV
14. StreamingMovies
15. Contract
16. PaperlessBilling
17. PaymentMethod
18. MonthlyCharges
19. TotalCharges
20. Churn
21. AverageMonthlySpend
22. IsNewCustomer
23. ServiceCount


In [20]:
working_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               7043 non-null   str    
 1   SeniorCitizen        7043 non-null   int64  
 2   Partner              7043 non-null   str    
 3   Dependents           7043 non-null   str    
 4   tenure               7043 non-null   int64  
 5   PhoneService         7043 non-null   str    
 6   MultipleLines        7043 non-null   str    
 7   InternetService      7043 non-null   str    
 8   OnlineSecurity       7043 non-null   str    
 9   OnlineBackup         7043 non-null   str    
 10  DeviceProtection     7043 non-null   str    
 11  TechSupport          7043 non-null   str    
 12  StreamingTV          7043 non-null   str    
 13  StreamingMovies      7043 non-null   str    
 14  Contract             7043 non-null   str    
 15  PaperlessBilling     7043 non-null   str    
 16 

## 15. Check missing values after preparation

The model-ready dataset should not contain unexpected missing values.

We check again after all cleaning and feature engineering.

In [21]:
missing_after_preparation = working_df.isnull().sum()

missing_after_preparation = (
    missing_after_preparation[
        missing_after_preparation > 0
    ]
    .sort_values(ascending=False)
)

if missing_after_preparation.empty:
    print(
        "No missing values remain in the prepared dataset."
    )
else:
    print("Missing values still present:")
    display(missing_after_preparation)

No missing values remain in the prepared dataset.


## 16. Check duplicate rows after preparation

Feature engineering should not create duplicate customer records.

We verify this before splitting the dataset.

In [22]:
duplicate_rows_after_preparation = (
    working_df.duplicated().sum()
)

print(
    f"Duplicate rows after preparation: "
    f"{duplicate_rows_after_preparation:,}"
)

Duplicate rows after preparation: 22


# Part E — Separate features and target

## 17. Create `X` and `y`

Machine-learning workflows normally separate:

- `X` → input features
- `y` → target

We do this only after cleaning and feature engineering decisions have been made.

In [23]:
X = working_df.drop(
    columns=[target_column]
)

y = working_df[target_column]

print(
    f"Number of features: {X.shape[1]}"
)

print(
    f"Number of target records: {y.shape[0]:,}"
)

Number of features: 22
Number of target records: 7,043


## 18. Identify numerical and categorical features

Different types of variables need different preprocessing.

### Numerical features

These can potentially be scaled.

### Categorical features

These need to be converted into a numerical representation before most machine-learning algorithms can use them.

In [24]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
for column in numerical_features:
    print(f"- {column}")

print()
print("Categorical features:")
for column in categorical_features:
    print(f"- {column}")

Numerical features:
- SeniorCitizen
- tenure
- MonthlyCharges
- TotalCharges
- AverageMonthlySpend
- IsNewCustomer
- ServiceCount

Categorical features:
- gender
- Partner
- Dependents
- PhoneService
- MultipleLines
- InternetService
- OnlineSecurity
- OnlineBackup
- DeviceProtection
- TechSupport
- StreamingTV
- StreamingMovies
- Contract
- PaperlessBilling
- PaymentMethod


C:\Users\Aman\AppData\Local\Temp\ipykernel_9848\3355591494.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


# Part F — Train/test split

## 19. Split before fitting preprocessing

This is a critical machine-learning practice.

We should not learn preprocessing information from the entire dataset before the split.

If we do that, information from the test set can indirectly influence the training process.

We therefore:

1. Split the data.
2. Fit preprocessing on training data only.
3. Apply the fitted preprocessing to validation/test data.

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training rows: {X_train.shape[0]:,}")
print(f"Test rows: {X_test.shape[0]:,}")

Training rows: 5,634
Test rows: 1,409


### Why `stratify=y`?

Churn is not evenly distributed between the two classes.

`stratify=y` keeps approximately the same churn proportion in both training and test sets.

This makes the split more representative of the original population.

In [26]:
print("Training target distribution:")
display(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("Percentage")
)

print("Test target distribution:")
display(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("Percentage")
)

Training target distribution:


,Percentage
Churn,
0,73.46
1,26.54


Test target distribution:


,Percentage
Churn,
0,73.46
1,26.54


# Part G — Build the preprocessing pipeline

## 20. Numerical preprocessing

For numerical features, we will use `StandardScaler`.

Scaling is useful for models that depend on feature magnitude, such as logistic regression, KNN, and SVM.

Tree-based models do not strictly require scaling, but keeping preprocessing in a controlled pipeline gives us a consistent interface across model experiments.

In [27]:
numerical_preprocessor = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        )
    ]
)

## 21. Categorical preprocessing

Categorical columns contain text such as:

- `Month-to-month`
- `One year`
- `Electronic check`

Machine-learning models need numerical representations.

We use **One-Hot Encoding**.

`handle_unknown="ignore"` is important for production: if a future customer contains a category that was not seen during training, the pipeline will not crash simply because of that unseen category.

In [28]:
categorical_preprocessor = Pipeline(
    steps=[
        (
            "one_hot_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

## 22. Combine numerical and categorical preprocessing

Now we combine both transformations into one `ColumnTransformer`.

This gives us a single preprocessing object that knows exactly how each feature type should be handled.

In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_preprocessor,
            numerical_features
        ),
        (
            "categorical",
            categorical_preprocessor,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 23. Fit preprocessing on training data only

This is where we actually learn the scaling parameters and category vocabulary.

Notice that we call `.fit()` using **`X_train` only**.

The test data is intentionally not used here.

In [30]:
preprocessor.fit(X_train)

print("Preprocessor fitted using training data only.")

Preprocessor fitted using training data only.


## 24. Transform training and test data

Now we apply the fitted transformations.

The test set is transformed using rules learned from the training set.

In [31]:
X_train_processed = preprocessor.transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Processed test shape:",
    X_test_processed.shape
)

Processed training shape: (5634, 48)
Processed test shape: (1409, 48)


## 25. Recover processed feature names

After one-hot encoding, the number of columns increases because categorical variables become multiple binary columns.

We retrieve the generated feature names so the transformed data remains interpretable.

In [32]:
processed_feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    f"Number of processed features: "
    f"{len(processed_feature_names):,}"
)

print()
print("First 20 processed feature names:")

for feature_name in processed_feature_names[:20]:
    print(feature_name)

Number of processed features: 48

First 20 processed feature names:
numerical__SeniorCitizen
numerical__tenure
numerical__MonthlyCharges
numerical__TotalCharges
numerical__AverageMonthlySpend
numerical__IsNewCustomer
numerical__ServiceCount
categorical__gender_Female
categorical__gender_Male
categorical__Partner_No
categorical__Partner_Yes
categorical__Dependents_No
categorical__Dependents_Yes
categorical__PhoneService_No
categorical__PhoneService_Yes
categorical__MultipleLines_No
categorical__MultipleLines_No phone service
categorical__MultipleLines_Yes
categorical__InternetService_DSL
categorical__InternetService_Fiber optic


## 26. Convert the processed arrays into DataFrames

Keeping the processed training and test data as DataFrames makes debugging and model interpretation easier.

The target remains separate as `y_train` and `y_test`.

In [33]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

print(
    "Processed training DataFrame shape:",
    X_train_processed_df.shape
)

print(
    "Processed test DataFrame shape:",
    X_test_processed_df.shape
)

Processed training DataFrame shape: (5634, 48)
Processed test DataFrame shape: (1409, 48)


# Part H — Final quality checks

## 27. Check for missing values in processed data

After encoding and scaling, there should be no unexpected missing values.

In [34]:
training_missing_values = (
    X_train_processed_df.isnull().sum().sum()
)

test_missing_values = (
    X_test_processed_df.isnull().sum().sum()
)

print(
    f"Missing values in processed training data: "
    f"{training_missing_values:,}"
)

print(
    f"Missing values in processed test data: "
    f"{test_missing_values:,}"
)

Missing values in processed training data: 0
Missing values in processed test data: 0


## 28. Final preparation summary

This is the checkpoint before modelling.

In [35]:
preparation_summary = {
    "Original rows": len(df),
    "Original columns": df.shape[1],
    "Final modelling features before encoding": X.shape[1],
    "Processed features after encoding": X_train_processed_df.shape[1],
    "Training rows": len(X_train),
    "Test rows": len(X_test),
    "Training missing values": training_missing_values,
    "Test missing values": test_missing_values,
    "Duplicate rows after preparation": duplicate_rows_after_preparation
}

preparation_summary

{'Original rows': 7043,
 'Original columns': 21,
 'Final modelling features before encoding': 22,
 'Processed features after encoding': 48,
 'Training rows': 5634,
 'Test rows': 1409,
 'Training missing values': np.int64(0),
 'Test missing values': np.int64(0),
 'Duplicate rows after preparation': np.int64(22)}

# 29. Final decisions

### Cleaning

- `TotalCharges` was converted to numeric.
- Problematic `TotalCharges` values were reconstructed using the customer's tenure and monthly charge where appropriate.
- The raw CSV remains untouched.

### Feature engineering

We created only a small number of business-oriented features:

- `AverageMonthlySpend`
- `IsNewCustomer`
- `ServiceCount`

The original service features were retained.

### Modelling preparation

- `customerID` is excluded from predictive features.
- `Churn` is encoded as `0/1`.
- Numerical and categorical features are handled separately.
- Categorical features are one-hot encoded.
- Numerical features are standardized.
- Train/test splitting uses stratification.
- Preprocessing is fitted using training data only.

### Leakage control

The preprocessing pipeline is fitted after the train/test split.

This prevents test-set information from influencing preprocessing parameters.

### Next step

The dataset is now ready for the **Modelling stage**.

The next notebook will compare several models using the same prepared data and establish a baseline before tuning.

We will focus on:

- baseline performance
- precision
- recall
- F1-score
- ROC-AUC
- model comparison
- business interpretation

> **We do not choose a model because it is popular. We choose it based on evidence from the problem and evaluation results.**